# Atelier Préparation de Données Images 

Contexte 
Une entreprise souhaite développer un système d’intelligence artificielle capable de reconnaître 
automatiquement le type de déchet présent sur une photographie afin d'améliorer le tri des 
déchets. 
Le modèle devra classer chaque image dans l'une des catégories suivantes : 
 cardboard : cartons ondulés, cartons plats, … 
 plastic : bouteilles, emballages plastiques... 
 paper : feuilles,  journaux... 
 glass : bouteilles et objets en verre... 
 metal : canettes, boîtes métalliques... 
 trash : emballages bonbons, tasses jetables, ... 
Le problème est que les images collectées proviennent de plusieurs sources. Elles ne sont donc pas 
homogènes : dimensions différentes ; formats différents ; images RGB et grayscale ; certaines images 
sont trop petites ; certaines images sont corrompues ; quelques images sont vides ; images 
dupliquées ; quelques images placées dans le mauvais dossier ; classes déséquilibrées. 
L'objectif de l'atelier est donc de construire un jeu de données images propre et homogène, prêt à 
être utilisé pour entraîner un modèle de Machine Learning ou de Deep Learning. 

Structure du projet 

In [1]:
import os
import hashlib
import glob
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
import shutil

# Définition des chemins
BASE_DIR = Path("../data")
RAW_DIR = BASE_DIR / "raw"
CLEANED_DIR = BASE_DIR / "cleaned"
REPORT_DIR = Path("../reports")
CLASSES = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']

# Partie 1 – Exploration du dataset

Développer un programme Python capable de récupérer, pour chaque image, son nom, sa classe, 
son format, son mode, sa largeur, sa hauteur, l’écart-type de ses pixels, son nombre de canaux et sa 
taille. 
NB : prendre en charge aussi les fichiers corrompus

In [2]:
data_list = []

for class_name in CLASSES:
    class_path = RAW_DIR / class_name
    for img_path in class_path.glob("*"):
        if img_path.is_file():
            file_size_kb = img_path.stat().st_size / 1024.0
            try:
                with Image.open(img_path) as img:
                    img.verify() # Vérification de l'intégrité
                with Image.open(img_path) as img:
                    img_arr = np.array(img)
                    width, height = img.size
                    mode = img.mode
                    fmt = img.format
                    channels = len(img.getbands())
                    std_dev = np.std(img_arr)
                    is_corrupted = False
            except Exception as e:
                width, height, mode, fmt, channels, std_dev = None, None, None, None, None, None
                is_corrupted = True

            data_list.append({
                'file_path': str(img_path),
                'file_name': img_path.name,
                'class': class_name,
                'format': fmt,
                'mode': mode,
                'width': width,
                'height': height,
                'channels': channels,
                'std_dev': std_dev,
                'size_kb': file_size_kb,
                'is_corrupted': is_corrupted
            })

df_meta = pd.DataFrame(data_list)
print(f"Total d'images recensées : {len(df_meta)}")
df_meta.head()

Total d'images recensées : 1032


,file_path,file_name,class,format,mode,width,height,channels,std_dev,size_kb,is_corrupted
0,..\data\raw\cardboard\cardboard1.jpg,cardboard1.jpg,cardboard,JPEG,RGB,512.0,384.0,3.0,40.586504,16.926758,False
1,..\data\raw\cardboard\cardboard10.jpg,cardboard10.jpg,cardboard,JPEG,RGB,512.0,384.0,3.0,42.577273,21.174805,False
2,..\data\raw\cardboard\cardboard100.jpg,cardboard100.jpg,cardboard,JPEG,RGB,512.0,384.0,3.0,46.121684,14.535156,False
3,..\data\raw\cardboard\cardboard101.jpg,cardboard101.jpg,cardboard,JPEG,RGB,512.0,384.0,3.0,72.264255,13.954102,False
4,..\data\raw\cardboard\cardboard102.jpg,cardboard102.jpg,cardboard,JPEG,RGB,512.0,384.0,3.0,48.389753,17.592773,False


# Partie 2 – Détecter les images corrompues 

Écrire et se servir d’une fonction qui détecte une image corrompue

Explication : Une image est corrompue si la bibliothèque PIL n'arrive pas à la lire ou à décoder son contenu binaire.

In [3]:
def check_corrupted(row):
    return row['is_corrupted']

df_meta['is_corrupted'] = df_meta.apply(check_corrupted, axis=1)
corrupted_files = df_meta[df_meta['is_corrupted']]

print(f"Nombre d'images corrompues : {len(corrupted_files)}")
corrupted_files[['file_name', 'class']]

Nombre d'images corrompues : 6


,file_name,class
147,cardboard83.jpg,cardboard
326,glass74.jpg,glass
446,metal48.jpg,metal
633,paper213.jpg,paper
791,plastic13.jpg,plastic
1004,trash3.jpg,trash


# Partie 3 – Détecter les images vides

Écrire et se servir d’une fonction qui détecte les images vides : image entièrement noire, image 
entièrement blanche ou image dont les pixels présentent très peu de variation.

Explication : Une image vide (tout noir, tout blanc) ou avec très peu de contenu a un écart-type de ses pixels proche de 0 ($\sigma < 5.0$).

In [4]:
def check_empty(row, std_threshold=5.0):
    if row['is_corrupted']:
        return False
    return row['std_dev'] < std_threshold

df_meta['is_empty'] = df_meta.apply(check_empty, axis=1)
empty_files = df_meta[df_meta['is_empty']]

print(f"Nombre d'images quasi vides : {len(empty_files)}")
empty_files[['file_name', 'class', 'std_dev']]

Nombre d'images quasi vides : 2


,file_name,class,std_dev
167,image-blanche-512x384.jpg,cardboard,1.574664
357,image-blanche-512x384.jpg,metal,1.574664
